In [5]:
import os
import re

# Ruta a la carpeta donde están los archivos
carpeta = 'SAFE_downloads'

# Expresión regular para capturar la fecha después de "MSIL1C_"
regex = re.compile(r'MSIL1C_(\d{8})T')

fechas = []

# Recorremos los archivos de la carpeta
for archivo in os.listdir(carpeta):
    if archivo.endswith('.zip'):
        match = regex.search(archivo)
        if match:
            fecha = match.group(1)  # formato YYYYMMDD
            # Convertimos a formato YYYY-MM-DD
            fechas.append(f"{fecha[:4]}-{fecha[4:6]}-{fecha[6:]}")

# Resultado
print(sorted(fechas))


['2016-08-09', '2016-09-08', '2017-06-30', '2018-02-20', '2018-03-07', '2018-05-16', '2018-06-20', '2018-07-10', '2018-08-29', '2018-10-03', '2018-11-07', '2019-03-12', '2019-06-25', '2019-07-10', '2019-07-30', '2019-08-14', '2019-09-28', '2019-10-03', '2019-11-27', '2020-02-20', '2020-03-11', '2020-12-21', '2021-01-05', '2021-04-20', '2021-05-20', '2021-06-14', '2021-08-13', '2021-11-11', '2021-11-26', '2021-12-01', '2022-02-24', '2022-07-14', '2022-07-29', '2022-08-03', '2022-09-07', '2022-09-27', '2023-01-10', '2023-03-01', '2023-03-16', '2023-04-20', '2023-05-25']


### Comprobar nubosidad para una fecha dada

In [7]:
from sentinelhub import SentinelHubCatalog, DataCollection, BBox, CRS, bbox_to_dimensions, SHConfig
from datetime import datetime
import configparser
from utils import get_access_token
from datetime import datetime, timedelta

/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
config_file = configparser.ConfigParser()
config_file.read("config.ini")

username = config_file["copernicus"]["username"]
password = config_file["copernicus"]["password"]

config = SHConfig()
config.sh_client_id = config_file["copernicus"]["client_id"] #"<CLIENT ID>"
config.sh_client_secret = config_file["copernicus"]["client_secret"] #<CLIENT SECRET>"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token" # Is it required?
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.save("cdse")
config = SHConfig("cdse")

In [9]:

def check_cloud_cover(access_token, time_interval, aoi, resolution, config): 
    # Definir el área de interés
    aoi_bbox = BBox(bbox=aoi, crs=CRS.WGS84)

    # Inicializar el catálogo
    catalog = SentinelHubCatalog(config=config)
    date_range = time_interval[0], time_interval[1]

    # Buscar imágenes en el intervalo de fechas
    search_iterator = catalog.search(
        DataCollection.SENTINEL2_L2A,
        bbox=aoi_bbox,
        time=date_range,
        fields={"include": ["id", "properties.eo:cloud_cover"], "exclude": ["properties.datetime"]},
    )
    results = list(search_iterator)

    # Filtrar resultados únicos por ID
    unique_results = {}
    for item in results:
        acquisition_id = item['id'].split('_T')[0]
        if acquisition_id not in unique_results:
            unique_results[acquisition_id] = item

    # Mostrar la nubosidad por fecha
    for item in unique_results.values():
        image_id = item['id']
        date = datetime.strptime(image_id.split('_')[2][:8], "%Y%m%d").date()
        cloud_cover = item["properties"]["eo:cloud_cover"]
        if cloud_cover > 20:
            print(f"🌩️ -- {date} - Cloud cover: {cloud_cover}%")
        else:
            print(f"     {date} - Cloud cover: {cloud_cover}%")


In [10]:
access_token = get_access_token(username, password)

target_dates = [
    '2016-08-09', '2016-09-08', '2017-06-30', '2018-02-20', '2018-03-07', 
    '2018-05-16', '2018-06-20', '2018-07-10', '2018-08-29', '2018-10-03',
    '2018-11-07', '2019-03-12', '2019-06-25', '2019-07-10', '2019-07-30', 
    '2019-08-14', '2019-09-28', '2019-10-03', '2019-11-27', '2020-02-20', 
    '2020-03-11', '2020-12-21', '2021-01-05', '2021-04-20', '2021-05-20',
    '2021-06-14', '2021-08-13', '2021-11-11', '2021-11-26', '2021-12-01', 
    '2022-02-24', '2022-07-14', '2022-07-29', '2022-08-03', '2022-09-07', 
    '2022-09-27', '2023-01-10', '2023-03-01', '2023-03-16', '2023-04-20', 
    '2023-05-25']
# '2016-08-09', '2016-09-08', '2016-10-28', '2017-01-26', '2017-06-30', 
# '2017-07-05', '2017-09-11', '2017-11-22', '2018-01-31', '2018-02-20', 
# '2018-03-07', '2018-05-11', '2018-05-16', '2018-06-20', '2018-07-10', 
# '2018-08-09', '2018-08-14', '2018-08-29', '2018-09-13', '2018-10-03', 
# '2018-11-07', '2019-02-20', '2019-03-07', '2019-03-12', '2019-06-25', 
# '2019-07-10', '2019-07-30', '2019-08-14', '2019-08-29', '2019-09-18', 
# '2019-09-28', '2019-10-03', '2019-11-07', '2019-11-27', '2019-12-02', 
# '2019-12-07', '2020-02-20', '2020-02-25', '2020-03-11', '2020-05-05', 
# '2020-05-20', '2020-05-25', '2020-06-02', '2020-08-13', '2020-12-21',
# '2021-01-05', '2021-01-13', '2021-01-20', '2021-02-24', '2021-04-20', 
# '2021-05-20', '2021-05-25', '2021-06-14', '2021-07-14', '2021-07-14', 
# '2021-08-03', '2021-08-13', '2021-09-02', '2021-09-27', '2021-10-07',
# '2021-11-11', '2021-11-26', '2021-12-01', '2021-12-21', '2022-02-24', 
# '2022-03-11', '2022-06-24', '2022-07-14', '2022-07-29', '2022-08-03', 
# '2022-09-07', '2022-09-27', '2023-01-10', '2023-01-20', '2023-01-25', 
# '2023-02-14', '2023-03-01', '2023-03-16', '2023-04-20', '2023-05-05', 
# '2023-05-25', '2023-07-19', '2023-09-07', '2023-09-27', '2023-10-17', 
# '2023-11-16'
# ]

slots = [((datetime.strptime(d, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d"), (datetime.strptime(d, "%Y-%m-%d") + timedelta(days=4)).strftime("%Y-%m-%d")) for d in target_dates]


for time_interval in slots: 

    check_cloud_cover(
        access_token=access_token,
        time_interval=time_interval,
        aoi= [-0.86, 37.65, -0.74, 37.8],  # coordenadas de ejemplo (W, S, E, N)
        resolution=10,
        config=config
    )

     2016-08-09 - Cloud cover: 0.0%
     2016-09-08 - Cloud cover: 3.42%
     2017-06-30 - Cloud cover: 0.39%
     2018-02-20 - Cloud cover: 10.16%
     2018-03-07 - Cloud cover: 13.42%
     2018-05-16 - Cloud cover: 0.1%
     2018-06-20 - Cloud cover: 0.01%
     2018-07-10 - Cloud cover: 2.77%
     2018-08-29 - Cloud cover: 2.7%
     2018-10-03 - Cloud cover: 3.44%
     2018-11-07 - Cloud cover: 0.61%
     2019-03-12 - Cloud cover: 1.02%
     2019-06-25 - Cloud cover: 5.14%
     2019-07-10 - Cloud cover: 0.07%
     2019-07-30 - Cloud cover: 2.61%
     2019-08-14 - Cloud cover: 2.39%
     2019-09-28 - Cloud cover: 11.32%
     2019-10-03 - Cloud cover: 16.39%
     2019-11-27 - Cloud cover: 11.67%
     2020-02-20 - Cloud cover: 11.57%
     2020-03-11 - Cloud cover: 0.0%
     2020-12-21 - Cloud cover: 2.4%
🌩️ -- 2021-01-05 - Cloud cover: 43.22%
     2021-04-20 - Cloud cover: 16.41%
     2021-05-20 - Cloud cover: 2.05%
     2021-06-14 - Cloud cover: 0.2%
     2021-08-13 - Cloud cover: 12.8

In [1]:
fechas_descargadas = [
    '2016-08-09', '2016-09-08', '2016-10-28', '2017-01-26', '2017-06-30', 
    '2017-07-05', '2017-11-22', '2018-01-31', '2018-02-20', '2018-03-07', 
    '2018-05-11', '2018-05-16', '2018-06-20', '2018-07-10', '2018-08-09', 
    '2018-08-14', '2018-08-29', '2018-09-13', '2018-10-03', '2018-11-07', 
    '2019-02-20', '2019-03-07', '2019-03-12', '2019-06-25', '2019-07-10', 
    '2019-07-30', '2019-08-14', '2019-08-29', '2019-09-18', '2019-09-28', 
    '2019-10-03', '2019-11-07', '2019-11-27', '2019-12-02', '2019-12-07',
    '2020-02-20', '2020-02-25', '2020-03-11', '2020-05-05', '2020-05-20',
    '2020-05-25', '2020-12-21', '2021-01-05', '2021-01-20', '2021-02-24', 
    '2021-04-20', '2021-05-20', '2021-05-25', '2021-06-14', '2021-08-03',
    '2021-08-13', '2021-09-02', '2021-09-27', '2021-10-07', '2021-11-11', 
    '2021-11-26', '2021-12-01', '2021-12-21', '2022-02-24', '2022-03-11', 
    '2022-06-24', '2022-07-14', '2022-07-29', '2022-08-03', '2022-09-07', 
    '2022-09-27', '2023-01-10', '2023-01-20', '2023-01-25', '2023-02-14', 
    '2023-03-01', '2023-03-16', '2023-04-20', '2023-05-05', '2023-05-25', 
    '2023-07-19', '2023-09-07', '2023-09-27', '2023-10-17', '2023-11-16'
]

fechas_filtro_1 = [
    '2018-01-31', '2018-08-09', '2018-08-14', '2019-03-07', '2019-09-18',
    '2019-11-07', '2020-02-25', '2020-05-20', '2020-05-05', '2021-07-14', 
    '2021-09-27', '2022-06-24', '2023-01-20', '2023-07-19', '2023-09-07', 
    '2023-09-27', '2023-11-16'
]

fechas_filtro_2 = [
    '2016-10-28', '2017-01-26', '2017-07-05', '2017-11-22', '2018-01-31', 
    '2018-05-11', '2018-09-13', '2019-02-20', '2019-03-07', '2019-08-29',
    '2019-09-18', '2019-11-07', '2019-12-02', '2019-12-07', '2020-02-25',
    '2020-05-05', '2020-05-25', '2021-01-20', '2021-02-24', '2021-05-25',
    '2021-07-14', '2021-08-03', '2021-09-02', '2021-09-27', '2021-10-07',
    '2021-12-21', '2022-03-11', '2022-06-24', '2023-01-20', '2023-01-25',
    '2023-02-14', '2023-05-05', '2023-07-19', '2023-09-07', '2023-09-27', 
    '2023-10-17', '2023-11-16'
]

In [2]:
fechas_disponibles = set(fechas_descargadas) - set(fechas_filtro_1) - set(fechas_filtro_2)

In [3]:
list(fechas_disponibles).sort()
print(len(fechas_disponibles))

41
